# 第7章

In [ ]:
!pip install pyro-ppl==1.9.0
!pip install pandas==2.2.1
!pip install pgmpy==0.1.25

## リスト7.1

In [ ]:
import pandas as pd
data_url = (
    "https://raw.githubusercontent.com/altdeep/causalAI/master/"
    "datasets/sidequests_and_purchases_obs.csv"
)
df = pd.read_csv(data_url)
summary = df.drop('User ID', axis=1).groupby(
    ["Side-quest Engagement"]
).agg(
    ['count', 'mean', 'std']
)
summary

## リスト7.2

In [ ]:
n1, n2 = summary['In-game Purchases']['count']
m1, m2 = summary['In-game Purchases']['mean']
s1, s2 =  summary['In-game Purchases']['std']
pooled_std = (s1**2 / n1 + s2**2 / n2) **.5
z_score = (m1 - m2) / pooled_std
abs(z_score) > 2.

## リスト7.3

In [ ]:
import pandas as pd
exp_data_url = (
    "https://raw.githubusercontent.com/altdeep/causalAI/master/"
    "datasets/sidequests_and_purchases_exp.csv"
)
df = pd.read_csv(exp_data_url)
summary = df.drop('User ID', axis=1).groupby(
    ["Side-quest Engagement"]
).agg(
    ['count', 'mean', 'std']
)
print(summary)

## リスト7.4

In [ ]:
n1, n2 = summary['In-game Purchases']['count']
m1, m2 = summary['In-game Purchases']['mean']
s1, s2 =  summary['In-game Purchases']['std']
pooled_std = (s1**2 / n1 + s2**2 / n2) **.5
z_score = (m1 - m2) / pooled_std
abs(z_score) > 2.

## リスト7.5

In [ ]:
import pandas as pd
full_obs_url = (
    "https://raw.githubusercontent.com/altdeep/causalAI/master/"
    "datasets/sidequests_and_purchases_full_obs.csv"
)
df = pd.read_csv(full_obs_url)

In [ ]:
membership_counts = df['Guild Membership'].value_counts()
dist_guild_membership = membership_counts / sum(membership_counts)
print(dist_guild_membership)

In [ ]:
dist_guild_membership.member

## リスト7.6

In [ ]:
member_subset = df[(df['Guild Membership'] == 'member')]
member_engagement_counts = (
    member_subset['Side-quest Engagement'].value_counts()
)
dist_engagement_member = (
    member_engagement_counts / sum(member_engagement_counts)
)
print(dist_engagement_member)

nonmember_subset = df[(df['Guild Membership'] == 'nonmember')]
nonmember_engagement_counts = (
    nonmember_subset['Side-quest Engagement'].value_counts()
)
dist_engagement_nonmember = (
    nonmember_engagement_counts / sum(nonmember_engagement_counts)
)
print(dist_engagement_nonmember)

## リスト7.7

In [ ]:
purchase_dist_nonmember_low_engagement = df[
    (df['Guild Membership'] == 'nonmember') &
    (df['Side-quest Engagement'] == 'low')
].drop(
  ['User ID', 'Side-quest Engagement', 'Guild Membership'], axis=1
).agg(['mean', 'std'])
print(round(purchase_dist_nonmember_low_engagement, 2))

purchase_dist_nonmember_high_engagement = df[
    (df['Guild Membership'] == 'nonmember') &
    (df['Side-quest Engagement'] == 'high')
].drop(
  ['User ID', 'Side-quest Engagement', 'Guild Membership'], axis=1
).agg(['mean', 'std'])
print(round(purchase_dist_nonmember_high_engagement, 2))

purchase_dist_member_low_engagement = df[
    (df['Guild Membership'] == 'member') &
    (df['Side-quest Engagement'] == 'low')
].drop(
  ['User ID', 'Side-quest Engagement', 'Guild Membership'], axis=1
).agg(['mean', 'std'])
print(round(purchase_dist_member_low_engagement, 2))

purchase_dist_member_high_engagement = df[
    (df['Guild Membership'] == 'member') &
    (df['Side-quest Engagement'] == 'high')
].drop(
  ['User ID', 'Side-quest Engagement', 'Guild Membership'], axis=1
).agg(['mean', 'std'])
print(round(purchase_dist_member_high_engagement, 2))

## リスト7.8

In [ ]:
import pyro
from torch import tensor
from pyro.distributions import Bernoulli, Normal

def model():
    p_member = tensor(0.5)
    is_guild_member = pyro.sample(
        "Guild Membership",
        Bernoulli(p_member)
    )
    p_engaged = (tensor(0.8)*is_guild_member +
                 tensor(.2)*(1-is_guild_member))
    is_highly_engaged = pyro.sample(
        "Side-quest Engagement",
        Bernoulli(p_engaged)
    )
    get_purchase_param = lambda param1, param2, param3, param4: (
        param1 * (1-is_guild_member) * (1-is_highly_engaged) +
        param2 * (1-is_guild_member) * (is_highly_engaged) +
        param3 * (is_guild_member)   * (1-is_highly_engaged) +
        param4 * (is_guild_member)   * (is_highly_engaged)
    )
    μ = get_purchase_param(37.95, 54.92, 223.71, 125.50)
    σ = get_purchase_param(23.80, 4.92, 5.30, 53.49)
    in_game_purchases = pyro.sample(
        "In-game Purchases",
        Normal(μ, σ)
    )
    guild_membership = "member" if is_guild_member else "nonmember"
    engagement = "high" if is_highly_engaged else "low"
    in_game_purchases = float(in_game_purchases)
    return guild_membership, engagement, in_game_purchases

In [ ]:
pyro.render_model(model)

In [ ]:
pyro.util.set_rng_seed(123)
simulated_observational_data = [model() for _ in range(1000)]
sim_full_obs_df = pd.DataFrame(
    simulated_observational_data,
    columns=["Guild Membership", "Side-quest Engagement", "In-Game Purchases"]
)
sim_obs_df = sim_full_obs_df.drop("Guild Membership", axis=1)
sim_obs_df.groupby(["Side-quest Engagement"]).agg(['count', 'mean', 'std'])

In [ ]:
df.drop(["Guild Membership", "User ID"], axis=1).groupby(["Side-quest Engagement"]).agg(['count', 'mean', 'std'])

## リスト7.9

In [ ]:
int_engaged_model = pyro.do(
    model,
    {"Side-quest Engagement": tensor(1.)}
)
int_unengaged_model = pyro.do(
    model,
    {"Side-quest Engagement": tensor(0.)}
)

## リスト7.10

In [ ]:
pyro.util.set_rng_seed(123)
simulated_experimental_data = [
    int_engaged_model() for _ in range(500)
] + [
    int_unengaged_model() for _ in range(500)
]
simulated_experimental_data = pd.DataFrame(
    simulated_experimental_data,
    columns=[
        "Guild Membership",
        "Side-quest Engagement",
        "In-Game Purchases"
    ]
)
sim_exp_df = simulated_experimental_data.drop(
    "Guild Membership", axis=1)
summary = sim_exp_df.groupby(
        ["Side-quest Engagement"]
    ).agg(
        ['count', 'mean', 'std']
    )
print(summary)

In [ ]:
exp_data_url = (
    "https://raw.githubusercontent.com/altdeep/causalAI/master/"
    "datasets/sidequests_and_purchases_exp.csv"
)
exp_df = pd.read_csv(exp_data_url)

summary = exp_df.drop('User ID', axis=1).groupby(
        ["Side-quest Engagement"]
    ).agg(
        ['count', 'mean', 'std']
    )
print(summary)

## リスト7.11

In [ ]:
from pgmpy.base import DAG

G = DAG([
    ('Guild Membership', 'Side-quest Engagement'),
    ('Side-quest Engagement', 'In-game Purchases'),
    ('Guild Membership', 'In-game Purchases')
])
G_int = G.do('Side-quest Engagement')

## リスト7.12

In [ ]:
import pylab as plt
import networkx as nx

pos = {
    'Guild Membership': [0.0, 1.0],
    'Side-quest Engagement': [-1.0, 0.0],
    'In-game Purchases': [1.0, -0.5]
}

ax = plt.subplot()
ax.margins(0.3)
nx.draw(G, ax=ax, pos=pos, node_size=3000,
        node_color='w', with_labels=True)
plt.show()

ax = plt.subplot()
ax.margins(0.3)
nx.draw(G_int, ax=ax, pos=pos,
        node_size=3000, node_color='w', with_labels=True)
plt.show()

## リスト7.13

In [ ]:
int_engaged_model = pyro.do(
    model,
    {"Side-quest Engagement": tensor(1.)}
)
int_unengaged_model = pyro.do(
    model,
    {"Side-quest Engagement": tensor(0.)}
)